In [ ]:
import pandas as pd
import scipy as sc
import numpy as np
import xgboost as xgb
import sklearn as sk
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_22 = pd.read_csv(r"D:\F1_ML\DATA\df_2022.csv")
df_23 = pd.read_csv(r"D:\F1_ML\DATA\df_2023.csv")
df_24 = pd.read_csv(r"D:\F1_ML\DATA\df_2024.csv")
df_25 = pd.read_csv(r"D:\F1_ML\DATA\df_2025.csv")
df_26 = pd.read_csv(r"D:\F1_ML\DATA\df_2026.csv")


In [ ]:
print(df_22.shape,
df_23.shape,
df_24.shape,
df_25.shape,
df_26.shape)

In [ ]:
df = pd.concat([df_22, df_23, df_24, df_25, df_26])

In [ ]:
df = df.drop(columns=['HeadshotUrl', 'TeamColor', 'BroadcastName', 'FirstName', 'LastName', 'DriverNumber', 'Abbreviation', 'TeamId', 'CountryCode'])

In [ ]:

quali_times = (
    df[df['form_of_race'] == 'Qualifying']
    [['year', 'gp', 'DriverId', 'Q1', 'Q2', 'Q3', 'Position']]
    .rename(columns={'Position': 'quali_position'})
)

races = df[df['form_of_race'] == 'Race'].drop(columns=['Q1', 'Q2', 'Q3']).copy()

df_train = races.merge(
    quali_times,
    on=['year', 'gp', 'DriverId'],
    how='left'
)


In [ ]:
# n_corners — liczba zakrętów
# overtaking_difficulty — subiektywna ocena 1-5
#   1 = bardzo łatwo wyprzedzać
#   5 = praktycznie niemożliwe
circuit_features = {
    'Bahrain Grand Prix':          {'n_corners': 15, 'overtaking_difficulty': 2},
    'Saudi Arabian Grand Prix':    {'n_corners': 27, 'overtaking_difficulty': 4},  
    'Australian Grand Prix':       {'n_corners': 14, 'overtaking_difficulty': 3},  
    'Japanese Grand Prix':         {'n_corners': 18, 'overtaking_difficulty': 3},  
    'Chinese Grand Prix':          {'n_corners': 16, 'overtaking_difficulty': 2},  
    'Miami Grand Prix':            {'n_corners': 19, 'overtaking_difficulty': 4},
    'Emilia Romagna Grand Prix':   {'n_corners': 19, 'overtaking_difficulty': 4},  
    'Monaco Grand Prix':           {'n_corners': 19, 'overtaking_difficulty': 5},
    'Canadian Grand Prix':         {'n_corners': 14, 'overtaking_difficulty': 2},  
    'Spanish Grand Prix':          {'n_corners': 14, 'overtaking_difficulty': 3},  
    'Austrian Grand Prix':         {'n_corners': 10, 'overtaking_difficulty': 2},  
    'British Grand Prix':          {'n_corners': 18, 'overtaking_difficulty': 2},  
    'Hungarian Grand Prix':        {'n_corners': 14, 'overtaking_difficulty': 4},  
    'Belgian Grand Prix':          {'n_corners': 19, 'overtaking_difficulty': 1},  
    'Dutch Grand Prix':            {'n_corners': 14, 'overtaking_difficulty': 4},  
    'Italian Grand Prix':          {'n_corners': 11, 'overtaking_difficulty': 1},  
    'Azerbaijan Grand Prix':       {'n_corners': 20, 'overtaking_difficulty': 3},  
    'Singapore Grand Prix':        {'n_corners': 19, 'overtaking_difficulty': 5},  
    'United States Grand Prix':    {'n_corners': 20, 'overtaking_difficulty': 2},  
    'Mexico City Grand Prix':      {'n_corners': 17, 'overtaking_difficulty': 3},
    'São Paulo Grand Prix':        {'n_corners': 15, 'overtaking_difficulty': 2},  
    'Las Vegas Grand Prix':        {'n_corners': 17, 'overtaking_difficulty': 3},
    'Qatar Grand Prix':            {'n_corners': 16, 'overtaking_difficulty': 3},  
    'Abu Dhabi Grand Prix':        {'n_corners': 16, 'overtaking_difficulty': 3},  
    'French Grand Prix':           {'n_corners': 15, 'overtaking_difficulty': 3},  
}


circuit_df = pd.DataFrame.from_dict(circuit_features, orient='index').reset_index()
circuit_df = circuit_df.rename(columns={'index': 'gp'})


df_train = df_train.merge(circuit_df, on='gp', how='left')



In [ ]:
df_train['grid_penalty'] = df_train['GridPosition'] - df_train['quali_position']


In [ ]:
f1_calendars = {
    2022: [
        'Bahrain Grand Prix',
        'Saudi Arabian Grand Prix',
        'Australian Grand Prix',
        'Emilia Romagna Grand Prix',
        'Miami Grand Prix',
        'Spanish Grand Prix',
        'Monaco Grand Prix',
        'Azerbaijan Grand Prix',
        'Canadian Grand Prix',
        'British Grand Prix',
        'Austrian Grand Prix',
        'French Grand Prix',
        'Hungarian Grand Prix',
        'Belgian Grand Prix',
        'Dutch Grand Prix',
        'Italian Grand Prix',
        'Singapore Grand Prix',
        'Japanese Grand Prix',
        'United States Grand Prix',
        'Mexico City Grand Prix',
        'São Paulo Grand Prix',
        'Abu Dhabi Grand Prix',
    ],
    2023: [
        'Bahrain Grand Prix',
        'Saudi Arabian Grand Prix',
        'Australian Grand Prix',
        'Azerbaijan Grand Prix',
        'Miami Grand Prix',
        'Monaco Grand Prix',           
        'Spanish Grand Prix',
        'Canadian Grand Prix',
        'Austrian Grand Prix',
        'British Grand Prix',
        'Hungarian Grand Prix',
        'Belgian Grand Prix',
        'Dutch Grand Prix',
        'Italian Grand Prix',
        'Singapore Grand Prix',
        'Japanese Grand Prix',
        'Qatar Grand Prix',
        'United States Grand Prix',
        'Mexico City Grand Prix',
        'São Paulo Grand Prix',
        'Las Vegas Grand Prix',
        'Abu Dhabi Grand Prix',
    ],
    2024: [
        'Bahrain Grand Prix',
        'Saudi Arabian Grand Prix',
        'Australian Grand Prix',
        'Japanese Grand Prix',
        'Chinese Grand Prix',
        'Miami Grand Prix',
        'Emilia Romagna Grand Prix',
        'Monaco Grand Prix',
        'Canadian Grand Prix',
        'Spanish Grand Prix',
        'Austrian Grand Prix',
        'British Grand Prix',
        'Hungarian Grand Prix',
        'Belgian Grand Prix',
        'Dutch Grand Prix',
        'Italian Grand Prix',
        'Azerbaijan Grand Prix',
        'Singapore Grand Prix',
        'United States Grand Prix',
        'Mexico City Grand Prix',
        'São Paulo Grand Prix',
        'Las Vegas Grand Prix',
        'Qatar Grand Prix',
        'Abu Dhabi Grand Prix',
    ],
    2025: [
        'Australian Grand Prix',
        'Chinese Grand Prix',
        'Japanese Grand Prix',
        'Bahrain Grand Prix',
        'Saudi Arabian Grand Prix',
        'Miami Grand Prix',
        'Emilia Romagna Grand Prix',
        'Monaco Grand Prix',
        'Spanish Grand Prix',
        'Canadian Grand Prix',
        'Austrian Grand Prix',
        'British Grand Prix',
        'Belgian Grand Prix',
        'Hungarian Grand Prix',
        'Dutch Grand Prix',
        'Italian Grand Prix',
        'Azerbaijan Grand Prix',
        'Singapore Grand Prix',
        'United States Grand Prix',
        'Mexico City Grand Prix',
        'São Paulo Grand Prix',
        'Las Vegas Grand Prix',
        'Qatar Grand Prix',
        'Abu Dhabi Grand Prix',
    ],
    2026: [
        'Australian Grand Prix',
        'Chinese Grand Prix',
        'Japanese Grand Prix',
        'Miami Grand Prix',
    ],
}

calendar_rows = [
    {'year': year, 'gp': gp, 'gp_round': i + 1}
    for year, gps in f1_calendars.items()
    for i, gp in enumerate(gps)
]
calendar_df = pd.DataFrame(calendar_rows)


df_train = df_train.merge(calendar_df, on=['year', 'gp'], how='left')



df_train['gp_round'] = df_train['gp_round'].astype('Int64')

In [ ]:
before = len(df_train)
df_train = df_train.dropna(subset=['gp_round']).copy()
after = len(df_train)

df_train['gp_round'] = df_train['gp_round'].astype(int)

In [ ]:
print(df_train.groupby('year')['gp_round'].nunique())

In [ ]:
df_train = df_train[df_train['form_of_race'] == 'Race'].copy()

df_train = df_train.sort_values(['year', 'gp_round', 'DriverId']).reset_index(drop=True)



In [ ]:
df_train['Position_num'] = pd.to_numeric(df_train['Position'], errors='coerce')

df_train['Position_filled'] = df_train['Position_num'].fillna(20)


In [ ]:
print(df_train['Status'].value_counts())

In [ ]:
finished_statuses_mask = (
    df_train['Status'].eq('Finished') |
    df_train['Status'].str.match(r'^\+\d+ Lap', na=False)
)

df_train['is_dnf'] = (~finished_statuses_mask).astype(int)

print(df_train[df_train['is_dnf'] == 1]['Status'].value_counts())

In [ ]:
finished_statuses_mask = (
    df_train['Status'].eq('Finished') |
    df_train['Status'].eq('Lapped') |
    df_train['Status'].str.match(r'^\+\d+ Lap', na=False)
)

df_train['is_dnf'] = (~finished_statuses_mask).astype(int)

print(df_train[df_train['is_dnf'] == 1]['Status'].value_counts())

In [ ]:
df_train['Position_num'] = pd.to_numeric(df_train['Position'], errors='coerce')

df_train['Position_for_form'] = df_train['Position_num'].copy()
df_train.loc[df_train['is_dnf'] == 1, 'Position_for_form'] = 20

df_train['Position_for_form'] = df_train['Position_for_form'].fillna(20)

In [ ]:
df_train = df_train.sort_values(['year', 'gp_round', 'DriverId']).reset_index(drop=True)

In [ ]:
df_train['driver_avg_pos_3'] = (
    df_train.groupby('DriverId')['Position_for_form']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

df_train['driver_avg_pos_5'] = (
    df_train.groupby('DriverId')['Position_for_form']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df_train['driver_dnf_rate_5'] = (
    df_train.groupby('DriverId')['is_dnf']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [ ]:
form_cols = ['driver_avg_pos_3', 'driver_avg_pos_5', 'driver_dnf_rate_5']

for col in form_cols:
    median_val = df_train[col].median()
    df_train[col] = df_train[col].fillna(median_val)
    
print(df_train[form_cols].describe())

In [ ]:
df_train.to_csv(r"D:\F1_ML\DATA\df_train_modified.csv", index=False)

In [ ]:
df_train.shape